# BoondManager — Test `pull_resume_attachment_list`

Each section is independent. Run sections 1 → 2 → 3 first, then any of 4–10.

## 1. Credentials

In [ ]:
import json, os

# secrets.json lives one level up from this notebook
_secrets_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'secrets.json')
with open(_secrets_path) as _f:
    _s = json.load(_f)

CLIENT_KEY        = _s['CLIENT_KEY']
CLIENT_TOKEN      = _s['CLIENT_TOKEN']
USER_TOKEN        = _s['USER_TOKEN']
HRFLOW_API_KEY    = _s['HRFLOW_API_SECRET']
HRFLOW_SOURCE_KEY = _s['HRFLOW_SOURCE_KEY']

CANDIDATE_STATES = None  # e.g. '0,1,2' — None means all
LIMIT = 3               # keep low while testing

print('Credentials loaded')
print(f'  CLIENT_KEY:        {CLIENT_KEY[:6]}...')
print(f'  HRFLOW_API_KEY:    {HRFLOW_API_KEY[:10]}...')
print(f'  HRFLOW_SOURCE_KEY: {HRFLOW_SOURCE_KEY[:10]}...')

## 2. Install dependencies

Installs the project in editable mode so pydantic, requests, etc. are available in this kernel.
Only needs to run once per kernel session.

In [ ]:
import subprocess, sys, os

# repo root is 6 levels above this notebook
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', '..', '..', '..'))
print('Installing from:', repo_root)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', repo_root, '--quiet'])
print('Done')

## 3. Setup

In [ ]:
import requests
from hrflow_connectors.v1.connectors.boondmanager.utils.jwt import auth_headers
from hrflow_connectors.v1.connectors.boondmanager.utils.api import (
    BOONDMANAGER_BASE_URL,
    REQUEST_TIMEOUT,
)

print('Setup OK — base URL:', BOONDMANAGER_BASE_URL)

## 4. List candidates

Mirrors the first step of `read_candidates_parsing`.

In [ ]:
list_params = {'sort': 'creationDate', 'page': 1}
if CANDIDATE_STATES:
    list_params['candidateStates'] = CANDIDATE_STATES

response = requests.get(
    url=f'{BOONDMANAGER_BASE_URL}/candidates',
    headers=auth_headers(USER_TOKEN, CLIENT_TOKEN, CLIENT_KEY),
    params=list_params,
    timeout=REQUEST_TIMEOUT,
)

print('Status:', response.status_code)
payload = response.json()
candidates = payload.get('data', [])
total = payload.get('meta', {}).get('totals', {}).get('rows', 0)
print(f'Total candidates in BoondManager: {total}')
print(f'Candidates on this page: {len(candidates)}')
for c in candidates[:5]:
    attrs = c.get('attributes', {})
    print(f"  id={c['id']}  name={attrs.get('firstName')} {attrs.get('lastName')}")

## 5. Fetch documents for a single candidate

In [ ]:
# Inspect the full JSON:API response for a candidate
# This reveals what relationships/links BoondManager exposes
cid = candidates[0]['id']

detail = requests.get(
    url=f'{BOONDMANAGER_BASE_URL}/candidates/{cid}',
    headers=auth_headers(USER_TOKEN, CLIENT_TOKEN, CLIENT_KEY),
    timeout=REQUEST_TIMEOUT,
)
data = detail.json().get('data', {})

print('=== TOP-LEVEL KEYS ===')
print(list(data.keys()))

print('\n=== RELATIONSHIPS ===')
for rel_name, rel_val in data.get('relationships', {}).items():
    links = rel_val.get('links', {})
    print(f'  {rel_name}: {links}')

print('\n=== LINKS ===')
print(data.get('links', {}))

print('\n=== FULL RESPONSE (truncated) ===')
print(json.dumps(data, indent=2)[:3000])

In [ ]:
# The resume ID comes from relationships.resumes in /information
# Now test how to download it
cid = candidates[0]['id']

r = requests.get(
    url=f'{BOONDMANAGER_BASE_URL}/candidates/{cid}/information',
    headers=auth_headers(USER_TOKEN, CLIENT_TOKEN, CLIENT_KEY),
    timeout=REQUEST_TIMEOUT,
)
data = r.json().get('data', {})
resume_data = data.get('relationships', {}).get('resumes', {}).get('data', [])
print(f'Resume entries in relationships: {resume_data}')

if not resume_data:
    print('No resumes for this candidate, trying next one...')
    for c2 in candidates[1:]:
        r2 = requests.get(
            url=f'{BOONDMANAGER_BASE_URL}/candidates/{c2["id"]}/information',
            headers=auth_headers(USER_TOKEN, CLIENT_TOKEN, CLIENT_KEY),
            timeout=REQUEST_TIMEOUT,
        )
        rd = r2.json().get('data', {}).get('relationships', {}).get('resumes', {}).get('data', [])
        if rd:
            cid = c2['id']
            resume_data = rd
            print(f'Found resume on candidate_id={cid}: {rd}')
            break

resume_id = resume_data[0]['id'] if resume_data else None
assert resume_id, 'No resume found in any candidate'

# Try different download URL patterns
for ep in [
    f'/documents/{resume_id}',
    f'/documents/{resume_id}/download',
    f'/candidates/{cid}/resumes/{resume_id}',
]:
    resp = requests.get(
        url=f'{BOONDMANAGER_BASE_URL}{ep}',
        headers=auth_headers(USER_TOKEN, CLIENT_TOKEN, CLIENT_KEY),
        timeout=REQUEST_TIMEOUT,
    )
    ct = resp.headers.get('Content-Type', '')
    preview = resp.content[:4] if resp.status_code == 200 else resp.text[:80]
    print(f'{resp.status_code}  GET {ep}  content-type={ct}  preview={preview}')


In [ ]:
# Scan candidates to find one that has a resume
# Resume ID lives in relationships.resumes from GET /candidates/{id}/information
SCAN_LIMIT = 20

candidate_with_resume = None
resume_id_found = None
for c2 in candidates[:SCAN_LIMIT]:
    cid = c2['id']
    r = requests.get(
        url=f'{BOONDMANAGER_BASE_URL}/candidates/{cid}/information',
        headers=auth_headers(USER_TOKEN, CLIENT_TOKEN, CLIENT_KEY),
        timeout=REQUEST_TIMEOUT,
    )
    if r.status_code != 200:
        attrs = c2.get('attributes', {})
        print(f'\u2717 id={cid}  name={attrs.get("firstName")} {attrs.get("lastName")}  status={r.status_code}')
        continue
    resume_entries = r.json().get('data', {}).get('relationships', {}).get('resumes', {}).get('data', [])
    attrs = c2.get('attributes', {})
    if resume_entries:
        print(f'\u2713 id={cid}  name={attrs.get("firstName")} {attrs.get("lastName")}  resumes={resume_entries}')
        candidate_with_resume = c2
        resume_id_found = resume_entries[-1]['id']
        break
    else:
        print(f'\u2717 id={cid}  name={attrs.get("firstName")} {attrs.get("lastName")}  no resumes')

if candidate_with_resume:
    test_candidate = candidate_with_resume
    candidate_id = candidate_with_resume['id']
    print(f'\nUsing candidate_id={candidate_id}  resume_id={resume_id_found}')
else:
    print(f'No candidates with resumes found in the first {SCAN_LIMIT}. Try increasing SCAN_LIMIT.')


In [ ]:
# Fetch /information for the selected candidate and extract the resume ID
r = requests.get(
    url=f'{BOONDMANAGER_BASE_URL}/candidates/{candidate_id}/information',
    headers=auth_headers(USER_TOKEN, CLIENT_TOKEN, CLIENT_KEY),
    timeout=REQUEST_TIMEOUT,
)
print('Status:', r.status_code)
info_data = r.json().get('data', {})
resume_entries = info_data.get('relationships', {}).get('resumes', {}).get('data', [])
print(f'Resume entries: {resume_entries}')
assert resume_entries, f'No resumes for candidate_id={candidate_id}'
resume_id = resume_entries[-1]['id']
print(f'Using resume_id={resume_id}')


## 6. Download the resume bytes

In [ ]:
assert documents, f'No documents for candidate_id={candidate_id} — try a different candidate'

resume_id = documents[-1].get('id')
print(f'Fetching document id={resume_id}')

resume_response = requests.get(
    url=f'{BOONDMANAGER_BASE_URL}/documents/{resume_id}',
    headers=auth_headers(USER_TOKEN, CLIENT_TOKEN, CLIENT_KEY),
    timeout=REQUEST_TIMEOUT,
)

print('Status:', resume_response.status_code)
print('Content-Type:', resume_response.headers.get('Content-Type'))
print(f'Bytes received: {len(resume_response.content)}')

if resume_response.content[:4] == b'%PDF':
    print('\u2713 Valid PDF')
else:
    print('\u26a0 Not a PDF — first 20 bytes:', resume_response.content[:20])

resume_bytes = resume_response.content

## 7. Build the parsing payload

In [ ]:
import base64

parsing_payload = {
    'source_key': HRFLOW_SOURCE_KEY,
    'profile_reference': str(candidate_id),
    'resume': {
        'raw': base64.b64encode(resume_bytes).decode('utf-8'),
        'content_type': 'application/pdf',
    },
    'tags': [],
    'metadatas': [],
}

print('Payload ready')
print(f"  reference:    {parsing_payload['profile_reference']}")
print(f'  resume bytes: {len(resume_bytes)}')

## 8. Send to HrFlow parsing API

In [ ]:
hrflow_response = requests.post(
    url='https://api.hrflow.ai/v1/profile/parsing/file',
    headers={
        'X-API-KEY': HRFLOW_API_KEY,
        'Content-Type': 'application/json',
    },
    json=parsing_payload,
    timeout=60,
)

print('Status:', hrflow_response.status_code)
print('Response:', hrflow_response.json())

## 9. Run the full warehouse loop (LIMIT candidates)

Calls `read_candidates_parsing` directly — same code path the connector uses.

In [ ]:
import logging
from hrflow_connectors.v1.connectors.boondmanager.warehouse import (
    ReadCandidatesParsingParameters,
    read_candidates_parsing,
)

adapter = logging.getLogger('boondmanager.test')
logging.basicConfig(level=logging.WARNING)

params = ReadCandidatesParsingParameters(
    client_key=CLIENT_KEY,
    client_token=CLIENT_TOKEN,
    user_token=USER_TOKEN,
    candidate_states=CANDIDATE_STATES,
    limit=LIMIT,
)

results = []
for candidate in read_candidates_parsing(adapter, params):
    resume = candidate.get('_resume_bytes')
    status = ('\u2713 ' + str(len(resume)) + ' bytes') if resume else '\u2717 missing'
    print(f"id={candidate['id']}  resume={status}")
    results.append(candidate)

print(f'\nTotal candidates with resumes: {len(results)}')

## 10. Run the full connector action (end-to-end)

In [ ]:
import hrflow_connectors

result = hrflow_connectors.BoondManager.pull_resume_attachment_list(
    workflow_id='test-notebook',
    action_parameters=dict(logics=[], format=None),
    origin_parameters=dict(
        client_key=CLIENT_KEY,
        client_token=CLIENT_TOKEN,
        user_token=USER_TOKEN,
        candidate_states=CANDIDATE_STATES,
        limit=LIMIT,
    ),
    target_parameters=dict(
        api_secret=HRFLOW_API_KEY,
        source_key=HRFLOW_SOURCE_KEY,
    ),
)

print(result)

---
## Bonus: Audit available fields vs what we currently map

Runs `GET /candidates/{id}/information` and `GET /opportunities/{id}/information`,
prints every attribute key and flags whether `format_candidate` / `format_opportunity` currently uses it.

In [ ]:
# Fields currently mapped by format_candidate
MAPPED_CANDIDATE = {
    'firstName', 'lastName', 'dateOfBirth', 'email1', 'phone1',
    'socialNetworks', 'address', 'postcode', 'town', 'country',
    'skills', 'languages', 'references', 'diplomas',
    'state', 'availability', 'globalEvaluation',
    'expertiseAreas', 'activityAreas', 'mobilityAreas',
    'numberOfActivePositionings', 'title',
    'creationDate', 'updateDate',
}

cid = candidates[0]['id']
r = requests.get(
    url=f'{BOONDMANAGER_BASE_URL}/candidates/{cid}/information',
    headers=auth_headers(USER_TOKEN, CLIENT_TOKEN, CLIENT_KEY),
    timeout=REQUEST_TIMEOUT,
)
attrs = r.json().get('data', {}).get('attributes', {})

print(f'=== /candidates/{cid}/information — all attribute keys ===')
for key, val in attrs.items():
    mapped = '✓ mapped' if key in MAPPED_CANDIDATE else '✗ NOT mapped'
    preview = str(val)[:60].replace('\n', ' ') if val not in (None, '', [], {}) else '(empty)'
    print(f'  {mapped:<12}  {key:<35}  {preview}')


In [ ]:
# Fields currently mapped by format_opportunity
MAPPED_OPPORTUNITY = {
    'title', 'creationDate', 'updateDate', 'place',
    'description', 'expertiseArea', 'origin', 'duration',
    'activityAreas', 'numberOfActivePositionings',
}

# Get first opportunity ID
opp_list = requests.get(
    url=f'{BOONDMANAGER_BASE_URL}/opportunities',
    headers=auth_headers(USER_TOKEN, CLIENT_TOKEN, CLIENT_KEY),
    timeout=REQUEST_TIMEOUT,
)
opps = opp_list.json().get('data', [])
assert opps, 'No opportunities found'
oid = opps[0]['id']

r = requests.get(
    url=f'{BOONDMANAGER_BASE_URL}/opportunities/{oid}/information',
    headers=auth_headers(USER_TOKEN, CLIENT_TOKEN, CLIENT_KEY),
    timeout=REQUEST_TIMEOUT,
)
attrs = r.json().get('data', {}).get('attributes', {})

print(f'=== /opportunities/{oid}/information — all attribute keys ===')
for key, val in attrs.items():
    mapped = '✓ mapped' if key in MAPPED_OPPORTUNITY else '✗ NOT mapped'
    preview = str(val)[:60].replace('\n', ' ') if val not in (None, '', [], {}) else '(empty)'
    print(f'  {mapped:<12}  {key:<35}  {preview}')
